Topic 180 | RAG - Document Loading & Data Ingestion

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a simple document
content = """
Artificial Intelligence (AI) is transforming the way people work, learn, and communicate.
It enables computers to perform tasks that normally require human intelligence, such as
understanding language, recognizing images, and making decisions. AI is widely used in
healthcare, education, finance, transportation, and many other industries. As AI continues
to evolve, it offers exciting opportunities to solve complex problems, improve productivity,
and create innovative solutions. However, it is also important to use AI responsibly and
consider its ethical and social impacts.
"""
with open("langchain_intro.txt", "w", encoding="utf-8") as f:
    f.write(content)


C:\Users\Ahsan Ali\AppData\Local\Temp\ipykernel_16180\2525223830.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\Ahsan Ali\Desktop\AI-course\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the document
loader = TextLoader("langchain_intro.txt", encoding="utf-8")
documents = loader.load()
print(f"loaded {len(documents)} document(s)")

loaded 1 document(s)


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
    )
splits = text_splitter.split_documents(documents)

In [4]:
print(f"splits into {len(splits)} chunks")
for i,split in enumerate(splits):
    print(f"chunk {i}: {split.page_content}")

splits into 7 chunks
chunk 0: Artificial Intelligence (AI) is transforming the way people work, learn, and communicate.
chunk 1: It enables computers to perform tasks that normally require human intelligence, such as
chunk 2: understanding language, recognizing images, and making decisions. AI is widely used in
chunk 3: healthcare, education, finance, transportation, and many other industries. As AI continues
chunk 4: to evolve, it offers exciting opportunities to solve complex problems, improve productivity,
chunk 5: and create innovative solutions. However, it is also important to use AI responsibly and
chunk 6: consider its ethical and social impacts.


Topic 181 | Building a RAG Pipeline

In [14]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS
# create embedings
embeding_function = HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("create embeding and indexing in fiass")
db = FAISS.from_documents(documents=splits, 
                          embedding=embeding_function)
print ("vector database create successfully")
print(f"store {len(splits)} vectors in the database")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 963.16it/s]


create embeding and indexing in fiass
vector database create successfully
store 7 vectors in the database


Download the llm and parpara the promote form RAG

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

#configration
model_id = "Qwen/Qwen2-0.5B-instruct"

#load model and tokenzer
print(f"loading model {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
    )
# create HF pipline
pip = pipeline(
    "text_generation",
    model=model,
    tokenizer = tokenizer,
    max_new_tokens=128,
    temperature =0.1,
    do_sample = True
)
# langchain llm wrapper
raw_llm = HuggingFacePipeline(pipeline=pip)

# define the qwen chat formate
def qwen_chat_format(input_dict):
    messages = [
        {
            "role":"system",
            "content": """
            you are helpfull AI assistant , answer questions strickly provide on context.
            do not use any external knowledge or makup infromation.
            if answer not in context respond exectly with "I don't know" and do not provide any other information.
            Always start your response with "Answer ": flowed by the answer or "i don't know" if answer not in context.
            """
        },
        {
            "role":"user",
            "content":f"""
            context:{input_dict["context"]}
            question:{input_dict["question"]}
            """
        }  
    ]
    formatted_prompt =tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return formatted_prompt
# generation function
def gereration_with_Qwen(formatted_prompt):
    response = raw_llm(formatted_prompt)
    
    generated = response.split(formatted_prompt)[-1].strip()
    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[1].strip()
    return generated
# create llm chain with formatting
llm_with_format = (
    RunnableLambda(qwen_chat_format)
    | RunnableLambda(gereration_with_Qwen)
    | StrOutputParser()
)
print("llm create successfully")

loading model Qwen/Qwen2-0.5B-instruct...


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`